In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import glob, os, yaml, sparse, itertools, subprocess, sys

import Bio.SeqUtils
import Bio.Data

from Bio import SeqIO
from Bio.Seq import Seq
from Bio import pairwise2
from Bio.pairwise2 import format_alignment

plt.rcParams['figure.dpi'] = 150
plt.rcParams['axes.titlepad'] = 10
import scipy.stats as st

# saliency_utils is in the utils_files directory
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "utils_files"))
from inSilicoMut_utils import *
from matplotlib.ticker import FormatStrFormatter
from sklearn.linear_model import LinearRegression

results_path = "/n/data1/hms/dbmi/farhat/Sanjana/CNN_results"
vcf_dir = "/n/scratch3/users/s/sak0914/annotated_VCF"

who_variants = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/WHO_catalog_clean.csv")
# df_mxf = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/MXF/data_for_model.csv")
# df_rif = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/single_drugs/RIF/data_for_model.csv")

h37Rv = SeqIO.read("/n/data1/hms/dbmi/farhat/Sanjana/GCF_000195955.2_ASM19595v2_genomic.gbff", "genbank")
h37Rv_genes = pd.read_csv("/n/data1/hms/dbmi/farhat/Sanjana/mycobrowser_h37rv_genes_v4.csv")

# Uncertain Significance Sites (in WHO Catalog) Investigation

## Focus on rpoB first, then move on to other genes

In [4]:
def get_dict_sites_and_categories(drug_abbr):
    
    who_variants_single_drug = who_variants.query("drug == @drug_abbr")
    
    sites_dict = {}
    
    for num in range(1, 6):
        
        sites = list(who_variants_single_drug.loc[who_variants["confidence"].str.contains(str(num))].genome_index.values)
        
        add_mut = []
        
        for i, mut in enumerate(sites):
            if "," in mut:
                for site in mut.split(","):
                    add_mut.append(site)

        sites += add_mut
        sites = [site for site in sites if "," not in site]
        sites = np.unique(sites).astype(int)

        sites_dict[num] = sites
        
    return sites_dict


# rif_sites_dict = get_dict_sites_and_categories("RIF")

In [5]:
rif_rpoBC_df = pd.read_csv("Rifampicin/rpoBC_saliency_results.csv")

uncertain_RIF_rpoB = who_variants.query("drug=='RIF' & confidence=='3) Uncertain significance' & mutation.str.contains('rpoB')").reset_index(drop=True)
del uncertain_RIF_rpoB["drug"]
del uncertain_RIF_rpoB["confidence"]

drop_mut = []
add_df = pd.DataFrame(columns=uncertain_RIF_rpoB.columns)

for i, row in uncertain_RIF_rpoB.iterrows():
    
    if "," not in row["genome_index"]:
        uncertain_RIF_rpoB.loc[i, "genome_index"] = int(row["genome_index"])
    else:
        split_sites = row["genome_index"].split(",")
        
        for site in split_sites:
            add_df = pd.concat([add_df, pd.DataFrame({"genome_index": int(site), 
                                                      "mutation": row["mutation"]
                                                     }, index=[-1])], axis=0)
            

        drop_mut.append(row["mutation"])
        
uncertain_RIF_rpoB = pd.concat([uncertain_RIF_rpoB.query("mutation not in @drop_mut"), add_df], axis=0).reset_index(drop=True)

In [7]:
rif_rpoBC_df

,Gene,Pos,Max,Min,Mean,Max_Sig,Min_Sig,WHO,Lineage_SNP
0,rpoBC,759611.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
1,rpoBC,759612.0,0.000069,0.0,9.241911e-09,1.0,0.0,0,0
2,rpoBC,759613.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
3,rpoBC,759614.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
4,rpoBC,759615.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...
7809,rpoBC,767316.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
7810,rpoBC,767317.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
7811,rpoBC,767318.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0
7812,rpoBC,767319.0,0.000000,0.0,0.000000e+00,0.0,0.0,0,0


In [12]:
rif_rpoBC_df["Pos"].values[0]

'759611.0'

In [8]:
rif_rpoBC_uncertain

,Gene,Pos,Max,Min,Mean,Max_Sig,Min_Sig,WHO,Lineage_SNP,POS,mutation


In [9]:
uncertain_RIF_rpoB

,genome_index,mutation
0,762280,rpoB_p.Glu825Gly
1,762819,rpoB_p.Phe1005Val
2,761448,rpoB_p.Phe548Leu
3,761813,rpoB_p.Phe669Leu
4,759690,rpoB_c.-117G>A
...,...,...
627,761432,rpoB_p.Ile542Lys
628,761452,rpoB_p.Val549Gly
629,761453,rpoB_p.Val549Gly
630,762357,rpoB_p.Asp851Pro


In [6]:
# keep only mutations that are in both 
rif_rpoBC_uncertain = rif_rpoBC_df.query("Max_Sig==1 | Min_Sig==1").merge(uncertain_RIF_rpoB, left_on="Pos", right_on="genome_index", how="inner")
print(f"{len(rif_rpoBC_uncertain['genome_index'].unique())} unique sites")

rif_rpoBC_uncertain = get_data_for_synthetic_VCF(rif_rpoBC_uncertain, "pos")
rif_rpoBC_uncertain.head()

0 unique sites


ValueError: Columns must be same length as key

In [ ]:
# create a header section
header = '##fileformat=VCFv4.1\n'
# header += '##reference=hg19\n'
header += "##contig=<ID=NC_000962.3,length=4411532>\n"
header += '#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tSample\n'

for mutation in rif_rpoBC_uncertain["mutation"].unique():
    
    variants_to_add = []
    
    for i, row in rif_rpoBC_uncertain.query("mutation==@mutation").reset_index(drop=True).iterrows():
        variants_to_add.append(['NC_000962.3', row["POS"], '.', row["REF"], row["ALT"], '.', 'PASS', '.', 'GT', '1/1'])

    # write the variants to a file
    with open(f'/n/data1/hms/dbmi/farhat/Sanjana/MIC_data/synthetic_VCFs/{row["mutation"].replace(".", "_").replace(">", "-")}.vcf', 'w') as f:
        f.write(header)
        for variant in variants_to_add:
            f.write('\t'.join(str(x) for x in variant) + '\n')

In [ ]:
rif_rpoBC_uncertain.query("mutation=='rpoB_p.Ile491Tyr'")

In [ ]:
plt.hist(rif_rpoBC_uncertain.Max)
plt.axvline(np.mean(rif_rpoBC_uncertain.Max), color='tomato', label=f"Mean = {np.round(np.mean(rif_rpoBC_uncertain.Max), 2)}")
sns.despine()
plt.legend()
plt.show()

In [ ]:
plt.hist(rif_rpoBC_uncertain.Min)
plt.axvline(np.mean(rif_rpoBC_uncertain.Min), color='tomato', label=f"Mean = {np.round(np.mean(rif_rpoBC_uncertain.Min), 2)}")
sns.despine()
plt.legend()
plt.show()